# Medical SFT Ablation

这个 notebook 用来做医疗领域的独立消融实验。

## 目标

1. 以 `Qwen/Qwen3.5-2B` 为原始基座
2. 从 `shibing624/medical` 中切出两个规模层次：`medical-400` 和 `medical-800`
3. 对比原始基座和不同医疗微调配置
4. 观察医疗风格、稳健性和是否容易过度自信

## 实验原则

- 医疗分支独立存在
- 不混入主通用指令集
- 先做评测，再决定是否值得训练
- 强调保守回答和不确定性表达
- 只做中文医疗 SFT，不碰奖励建模和 RL


## 1. 实验矩阵

第一版建议至少比较这些版本：

1. 原始基座 `Qwen/Qwen3.5-2B`
2. `medical-400`
3. `medical-800`
4. `medical-800 + lower lr`
5. `medical-800 + higher rank`


## 2. 数据来源

数据源：`shibing624/medical`

优先使用其中的 `finetune` 部分，并按官方 split 或抽样方式切出训练 / 验证 / 测试。

参考：
- [shibing624/medical 数据集](https://huggingface.co/datasets/shibing624/medical)
- [MedicalGPT 数据集说明](https://github.com/shibing624/MedicalGPT/wiki/%E6%95%B0%E6%8D%AE%E9%9B%86)

已确认该数据集包含 `finetune/train_zh_0.json`、`finetune/valid_zh_0.json`、`finetune/test_zh_0.json` 等文件。


## 3. 评测重点

医疗场景不能只看 loss，重点看：

1. 是否遵循指令
2. 是否比基座更稳
3. 是否过度自信
4. 是否会乱诊断
5. 是否会提示进一步就医
6. 是否结构清晰
7. 是否在不确定时保持保守


## 4. 导入依赖

建议使用的训练栈：

- `transformers`
- `datasets`
- `peft`
- `accelerate`
- `matplotlib`
- `seaborn`
- `pandas`

当前微调方式：`PEFT + LoRA`。


In [ ]:
from pathlib import Path
import json
import re
from dataclasses import dataclass, asdict

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, PeftModel

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 180)


## 5. 路径与基础配置

这个 notebook 独立于主线通用指令项目。

当前基座保持不变：`Qwen/Qwen3.5-2B`。


In [ ]:
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == 'notebooks':
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = Path('/Users/yezibin/Project/learn-ft-and-rl/qwen-instruction-sft-project/qwen-instruction-sft')

DATA_DIR = PROJECT_DIR / 'data'
RUNS_DIR = PROJECT_DIR / 'runs'
MEDICAL_DIR = DATA_DIR / 'medical'
RAW_MEDICAL_DIR = MEDICAL_DIR / 'raw'
TRAIN_MEDICAL_DIR = MEDICAL_DIR / 'train'
VALID_MEDICAL_DIR = MEDICAL_DIR / 'valid'
TEST_MEDICAL_DIR = MEDICAL_DIR / 'test'

BASE_MODEL = 'Qwen/Qwen3.5-2B'
RAW_SOURCE = 'shibing624/medical'
RAW_CONFIG = 'finetune'
MAX_LENGTH = 1024
MAX_NEW_TOKENS = 128
SEED = 42

PROJECT_DIR


## 6. 医疗数据切片计划

第一版只做两个规模层次：

- `medical-400`
- `medical-800`

训练 / 验证 / 测试建议固定为：

- train: 400 或 800
- valid: 100
- test: 100

这样可以直接比较规模扩展是否带来收益。


In [ ]:
medical_experiments = [
    {
        'name': 'medical-400',
        'train_size': 400,
        'valid_size': 100,
        'test_size': 100,
        'output_dir': str(RUNS_DIR / 'medical' / 'medical-400'),
        'lora_r': 8,
        'lora_alpha': 16,
        'lora_dropout': 0.05,
        'learning_rate': 2e-4,
        'num_train_epochs': 3,
    },
    {
        'name': 'medical-800',
        'train_size': 800,
        'valid_size': 100,
        'test_size': 100,
        'output_dir': str(RUNS_DIR / 'medical' / 'medical-800'),
        'lora_r': 8,
        'lora_alpha': 16,
        'lora_dropout': 0.05,
        'learning_rate': 2e-4,
        'num_train_epochs': 3,
    },
    {
        'name': 'medical-800-lower-lr',
        'train_size': 800,
        'valid_size': 100,
        'test_size': 100,
        'output_dir': str(RUNS_DIR / 'medical' / 'medical-800-lower-lr'),
        'lora_r': 8,
        'lora_alpha': 16,
        'lora_dropout': 0.05,
        'learning_rate': 1e-4,
        'num_train_epochs': 3,
    },
    {
        'name': 'medical-800-higher-rank',
        'train_size': 800,
        'valid_size': 100,
        'test_size': 100,
        'output_dir': str(RUNS_DIR / 'medical' / 'medical-800-higher-rank'),
        'lora_r': 16,
        'lora_alpha': 32,
        'lora_dropout': 0.05,
        'learning_rate': 2e-4,
        'num_train_epochs': 3,
    },
]

pd.DataFrame(medical_experiments)


## 7. 加载原始医疗数据

我们优先读取 `finetune` 配置，再自动识别 train / valid / test 对应的 split。


In [ ]:
def load_medical_dataset():
    ds = load_dataset(RAW_SOURCE, RAW_CONFIG)
    return ds


def resolve_split(ds_dict, candidates):
    keys = list(ds_dict.keys())
    lower_keys = {k.lower(): k for k in keys}
    for cand in candidates:
        cand = cand.lower()
        for key in keys:
            if cand in key.lower():
                return ds_dict[key]
    raise KeyError(f'Cannot resolve split from keys={keys} with candidates={candidates}')


def preview_dataset(ds):
    if hasattr(ds, 'keys'):
        return list(ds.keys())
    return type(ds).__name__

# raw_ds = load_medical_dataset()
# print(preview_dataset(raw_ds))


## 8. 医疗样本标准化

把 `instruction / input / output` 统一转换成当前项目使用的 `messages + metadata`。


In [ ]:
def build_messages(instruction, input_text, output_text=None):
    instruction = (instruction or '').strip()
    input_text = (input_text or '').strip()
    output_text = (output_text or '').strip() if output_text is not None else None
    user_content = instruction if not input_text else f'{instruction}

{input_text}'
    messages = [
        {
            'role': 'system',
            'content': '你是一个专业、谨慎、遵循医学安全原则的中文医疗助手。回答时优先保持保守，不确定时应明确说明并建议进一步咨询医生。',
        },
        {'role': 'user', 'content': user_content},
    ]
    if output_text is not None:
        messages.append({'role': 'assistant', 'content': output_text})
    return messages


def normalize_medical_row(row):
    instruction = row.get('instruction', '')
    input_text = row.get('input', '')
    output_text = row.get('output', '')
    return {
        'messages': build_messages(instruction, input_text, output_text),
        'metadata': {
            'source': 'shibing624/medical',
            'task_type': 'medical_sft',
            'language': 'zh',
        },
        'instruction': instruction,
        'input': input_text,
        'output': output_text,
    }


def to_dataframe(ds):
    return pd.DataFrame([normalize_medical_row(x) for x in ds])


## 9. 切分 400 / 800 数据集并落盘

这里会把抽样后的切片保存到本地，方便后续重复实验时直接读取。


In [ ]:
def sample_split(ds, n, seed=SEED):
    n = min(n, len(ds))
    return ds.shuffle(seed=seed).select(range(n))


def save_jsonl(records, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in records:
            f.write(json.dumps(row, ensure_ascii=False) + '
')


def materialize_medical_version(raw_ds, version_name, train_size, valid_size, test_size):
    train_split = resolve_split(raw_ds, ['train_zh', 'train'])
    valid_split = resolve_split(raw_ds, ['valid_zh', 'valid', 'validation'])
    test_split = resolve_split(raw_ds, ['test_zh', 'test'])

    train_rows = sample_split(train_split, train_size, seed=SEED)
    valid_rows = sample_split(valid_split, valid_size, seed=SEED)
    test_rows = sample_split(test_split, test_size, seed=SEED)

    train_records = [normalize_medical_row(x) for x in train_rows]
    valid_records = [normalize_medical_row(x) for x in valid_rows]
    test_records = [normalize_medical_row(x) for x in test_rows]

    save_jsonl(train_records, TRAIN_MEDICAL_DIR / f'{version_name}.jsonl')
    save_jsonl(valid_records, VALID_MEDICAL_DIR / f'{version_name}.jsonl')
    save_jsonl(test_records, TEST_MEDICAL_DIR / f'{version_name}.jsonl')

    return train_records, valid_records, test_records

# raw_ds = load_medical_dataset()
# medical_400_train, medical_400_valid, medical_400_test = materialize_medical_version(raw_ds, 'medical-400', 400, 100, 100)
# medical_800_train, medical_800_valid, medical_800_test = materialize_medical_version(raw_ds, 'medical-800', 800, 100, 100)


## 10. 数据预览

抽样后先看几条样本，确认字段和格式符合预期。


In [ ]:
def preview_records(records, n=3):
    return pd.DataFrame(records[:n])

# preview_records(medical_400_train)


## 11. tokenizer 和 chat template

医疗分支也必须与 Qwen 的 chat template 对齐。


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# preview_text = tokenizer.apply_chat_template(medical_400_train[0]['messages'], tokenize=False, add_generation_prompt=False)
# print(preview_text)


## 12. 转成训练文本和 Dataset

第一版使用标准 causal language modeling。


In [ ]:
def to_chat_text(record):
    return {
        'text': tokenizer.apply_chat_template(
            record['messages'],
            tokenize=False,
            add_generation_prompt=False,
        ),
        'metadata': record['metadata'],
        'instruction': record['instruction'],
        'input': record['input'],
        'output': record['output'],
    }


def build_dataset(records):
    return Dataset.from_list([to_chat_text(r) for r in records])


def tokenize_batch(batch):
    out = tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )
    out['labels'] = out['input_ids'].copy()
    return out

# train_ds_400 = build_dataset(medical_400_train)
# valid_ds_400 = build_dataset(medical_400_valid)
# train_ds_400 = train_ds_400.map(tokenize_batch, batched=True)
# valid_ds_400 = valid_ds_400.map(tokenize_batch, batched=True)


## 13. 实验配置

这里把医疗实验和主项目一样做成可比较的配置对象。


In [ ]:
@dataclass
class MedicalExperimentConfig:
    name: str
    train_size: int
    valid_size: int
    test_size: int
    output_dir: str
    lora_r: int
    lora_alpha: int
    lora_dropout: float
    learning_rate: float
    num_train_epochs: int

medical_configs = [MedicalExperimentConfig(**cfg) for cfg in medical_experiments]
pd.DataFrame([asdict(cfg) for cfg in medical_configs])


## 14. 构建训练器

这一部分使用 `PEFT + LoRA`。医疗分支仍然不做 full fine-tuning。


In [ ]:
def build_medical_model_and_targets(model_name=BASE_MODEL):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        device_map='auto',
        torch_dtype='auto',
    )
    lora_targets = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj', 'gate_proj']
    return model, lora_targets


def build_medical_trainer(exp_cfg: MedicalExperimentConfig, train_dataset, valid_dataset):
    model, lora_targets = build_medical_model_and_targets()
    peft_cfg = LoraConfig(
        r=exp_cfg.lora_r,
        lora_alpha=exp_cfg.lora_alpha,
        lora_dropout=exp_cfg.lora_dropout,
        target_modules=lora_targets,
        bias='none',
        task_type='CAUSAL_LM',
    )
    model = get_peft_model(model, peft_cfg)
    args = TrainingArguments(
        output_dir=exp_cfg.output_dir,
        learning_rate=exp_cfg.learning_rate,
        num_train_epochs=exp_cfg.num_train_epochs,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        logging_steps=5,
        report_to='none',
        bf16=True,
        seed=SEED,
    )
    return Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    )


## 15. 训练与评测执行

这一步会针对每个 medical 配置执行训练，保存日志和 eval 结果。


In [ ]:
def run_medical_experiment(exp_cfg: MedicalExperimentConfig, train_records, valid_records):
    run_dir = Path(exp_cfg.output_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    train_ds = build_dataset(train_records).map(tokenize_batch, batched=True)
    valid_ds = build_dataset(valid_records).map(tokenize_batch, batched=True)
    trainer = build_medical_trainer(exp_cfg, train_ds, valid_ds)

    train_output = trainer.train()
    eval_output = trainer.evaluate()

    history = pd.DataFrame(trainer.state.log_history)
    history['experiment'] = exp_cfg.name
    history.to_csv(run_dir / 'log_history.csv', index=False)

    with open(run_dir / 'eval_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(eval_output, f, ensure_ascii=False, indent=2)

    trainer.save_model(run_dir)
    return history, eval_output

# 运行示例：
# raw_ds = load_medical_dataset()
# train_400, valid_400, test_400 = materialize_medical_version(raw_ds, 'medical-400', 400, 100, 100)
# history_400, eval_400 = run_medical_experiment(medical_configs[0], train_400, valid_400)


## 16. 评测关注点

医疗任务的重点不是“回答得很像”，而是：

1. 是否结构清晰
2. 是否保守
3. 是否避免乱诊断
4. 是否会建议进一步就医
5. 是否比原始基座更稳


In [ ]:
def contains_any(text, keywords):
    return any(k in text for k in keywords)


def medical_eval_checks(text):
    return {
        'has_caution_language': contains_any(text, ['建议', '就医', '咨询', '进一步', '如果', '无法判断', '请及时']),
        'has_uncertainty_language': contains_any(text, ['无法确定', '不确定', '可能', '需要结合', '不能替代']),
        'length': len(text),
    }


sample_eval_cases = [
    {
        'task_type': 'medical_basics',
        'prompt': '请解释高血压的常见风险因素。',
    },
    {
        'task_type': 'medical_caution',
        'prompt': '如果出现持续不适但不清楚原因，应该怎么做？',
    },
    {
        'task_type': 'medical_structure',
        'prompt': '请用三点说明感冒时一般需要注意哪些常识。',
    },
    {
        'task_type': 'medical_referral',
        'prompt': '什么时候应该尽快去医院就诊？',
    },
]

pd.DataFrame(sample_eval_cases)


## 17. baseline 和 adapter 推理

这里会生成：

- 原始基座输出
- 每个医疗实验输出
- 规则检查结果

这样可以直接做样本级对比。


In [ ]:
def build_generation_messages(user_prompt):
    return [
        {
            'role': 'system',
            'content': '你是一个专业、谨慎、遵循医学安全原则的中文医疗助手。回答时优先保持保守，不确定时应明确说明并建议进一步咨询医生。',
        },
        {'role': 'user', 'content': user_prompt},
    ]


def get_model_device(model):
    return next(model.parameters()).device


def load_base_generation_model(model_name=BASE_MODEL):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        device_map='auto',
        torch_dtype='auto',
    )
    model.eval()
    return model


def generate_with_model(model, user_prompt, max_new_tokens=MAX_NEW_TOKENS):
    messages = build_generation_messages(user_prompt)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    device = get_model_device(model)
    inputs = tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def load_lora_model(adapter_dir, base_model_name=BASE_MODEL):
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        trust_remote_code=True,
        device_map='auto',
        torch_dtype='auto',
    )
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()
    return model


def adapter_exists(exp_cfg):
    adapter_dir = Path(exp_cfg.output_dir)
    return (adapter_dir / 'adapter_config.json').exists() or (adapter_dir / 'adapter_model.safetensors').exists()


## 18. 样本级输出对比

对比维度：

- baseline 输出
- 各实验输出
- 规则检查结果

医疗场景优先看保守性和是否提示就医。


In [ ]:
def build_sample_comparison_table(base_model, experiments, sample_eval_cases):
    rows = []
    for case in sample_eval_cases:
        base_output = generate_with_model(base_model, case['prompt'])
        base_eval = medical_eval_checks(base_output)
        row = {
            'task_type': case['task_type'],
            'prompt': case['prompt'],
            'baseline_output': base_output,
            'baseline_has_caution': base_eval['has_caution_language'],
            'baseline_has_uncertainty': base_eval['has_uncertainty_language'],
            'baseline_length': base_eval['length'],
        }
        for exp in experiments:
            if adapter_exists(exp):
                model = load_lora_model(exp.output_dir)
                out = generate_with_model(model, case['prompt'])
                ev = medical_eval_checks(out)
                row[f'{exp.name}_output'] = out
                row[f'{exp.name}_has_caution'] = ev['has_caution_language']
                row[f'{exp.name}_has_uncertainty'] = ev['has_uncertainty_language']
                row[f'{exp.name}_length'] = ev['length']
            else:
                row[f'{exp.name}_output'] = '<adapter not found>'
                row[f'{exp.name}_has_caution'] = None
                row[f'{exp.name}_has_uncertainty'] = None
                row[f'{exp.name}_length'] = None
        rows.append(row)
    return pd.DataFrame(rows)

# base_model_for_eval = load_base_generation_model()
# sample_comparison_df = build_sample_comparison_table(base_model_for_eval, medical_configs, sample_eval_cases)
# sample_comparison_df.to_csv(RUNS_DIR / 'medical' / 'sample_comparison.csv', index=False)
# sample_comparison_df


## 19. 自动实验汇总表

把训练配置和评测结果汇总成一张表，便于比较 `medical-400` 和 `medical-800`。


In [ ]:
def load_eval_metrics(run_dir):
    path = Path(run_dir) / 'eval_metrics.json'
    if path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    return {}


def load_histories(run_dirs):
    frames = []
    for run_dir in run_dirs:
        csv_path = Path(run_dir) / 'log_history.csv'
        if csv_path.exists():
            frames.append(pd.read_csv(csv_path))
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def summarize_experiments(experiments, history_df):
    rows = []
    for exp in experiments:
        row = asdict(exp)
        metrics = load_eval_metrics(exp['output_dir']) if isinstance(exp, dict) else load_eval_metrics(exp.output_dir)
        row['eval_loss'] = metrics.get('eval_loss')
        exp_name = exp['name'] if isinstance(exp, dict) else exp.name
        if not history_df.empty and 'experiment' in history_df.columns:
            exp_hist = history_df[history_df['experiment'] == exp_name]
        else:
            exp_hist = pd.DataFrame()
        if not exp_hist.empty and 'loss' in exp_hist.columns:
            loss_hist = exp_hist.dropna(subset=['loss'])
            row['final_train_loss'] = loss_hist['loss'].iloc[-1] if not loss_hist.empty else None
        else:
            row['final_train_loss'] = None
        rows.append(row)
    return pd.DataFrame(rows)

# history_df = load_histories([cfg['output_dir'] for cfg in medical_experiments])
# medical_summary_df = summarize_experiments(medical_experiments, history_df)
# medical_summary_df


## 20. 图表输出

至少保存：

1. training loss comparison
2. eval loss comparison
3. experiment summary table
4. sample comparison table


In [ ]:
# if not history_df.empty:
#     if 'loss' in history_df.columns:
#         train_loss_df = history_df.dropna(subset=['loss'])
#         if not train_loss_df.empty:
#             plt.figure(figsize=(10, 5))
#             sns.lineplot(data=train_loss_df, x='step', y='loss', hue='experiment')
#             plt.title('Medical Training Loss Comparison')
#             plt.tight_layout()
#             plt.savefig(RUNS_DIR / 'medical' / 'training_loss_comparison.png', dpi=150)
#             plt.show()
#
#     if 'eval_loss' in history_df.columns:
#         eval_loss_df = history_df.dropna(subset=['eval_loss'])
#         if not eval_loss_df.empty:
#             plt.figure(figsize=(10, 5))
#             sns.lineplot(data=eval_loss_df, x='epoch', y='eval_loss', hue='experiment', marker='o')
#             plt.title('Medical Eval Loss Comparison')
#             plt.tight_layout()
#             plt.savefig(RUNS_DIR / 'medical' / 'eval_loss_comparison.png', dpi=150)
#             plt.show()
#
# medical_summary_df.to_csv(RUNS_DIR / 'medical' / 'experiment_summary.csv', index=False)
# sample_comparison_df.to_csv(RUNS_DIR / 'medical' / 'sample_comparison.csv', index=False)


## 21. 总结

这份 medical notebook 现在包含：

- 数据加载
- 400 / 800 切片
- 统一 messages 转换
- LoRA 训练器
- 原始基座对比
- 样本级安全评测
- 实验汇总与图表输出

下一步就是直接运行它，而不是继续补结构。
